# From your original study to an auditable ML project

Run from the repository or notebooks folder. Read each question before running its cell. This notebook uses the actual matched data; it does not claim clinical validation.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
from data_io import aligned_rows
rows = [r for r in aligned_rows(ROOT/'data/source') if r['sample_type']=='01']
print('Primary tumors:', len(rows))
print({k: rows[0][k] for k in ['sample_id','patient_id','MCM5','mir885_activity_proxy']})


## 1. What do the columns mean?

`mir885_activity_proxy` is the negative of a target-gene score, not a direct assay. MCM5 is in that target signature. Why does that complicate interpreting the correlation?

In [ ]:
import numpy as np
X = np.array([[r['MCM5'],r['mir885_activity_proxy']] for r in rows])
print('Correlation:', np.corrcoef(X.T)[0,1])
assert len({r['patient_id'] for r in rows}) == len(rows)


## 2. Fit one understandable baseline

Change `k` to 3 only after explaining what K-means optimizes. The scaler belongs to the fitted pipeline.

In [ ]:
from modeling import fit_kmeans
k = 2
model = fit_kmeans(X, k, seed=52)
labels = model.predict(X)
print('Cluster sizes:', np.bincount(labels))
for cluster in range(k):
    print(cluster, 'mean MCM5, proxy:', X[labels==cluster].mean(axis=0))


## 3. Read the experiment rather than trusting the picture

The saved run compares three feature sets and a continuous Gaussian reference. Does the size of the difference justify a subtype claim?

In [ ]:
import json
summary = json.loads((ROOT/'results/summary.json').read_text())
for name, experiment in summary['experiments'].items():
    m = experiment['candidate_metrics']
    print(name, {key: round(m[key],3) for key in ['silhouette','null_silhouette_mean','subsample_ari_mean']})


## 4. Inspect the figures

PCA of two variables into two components is a rotation, not dimension reduction. t-SNE is visualization only.

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(ROOT/'results/two_marker_selection.svg')))
display(SVG(filename=str(ROOT/'results/pathways_pca.svg')))


## 5. Your next experiment

Write your hypothesis here before changing code. Recommended: recompute the miRNA target score without MCM5 using the original-study repository. Record your expected result, one changed variable, metric, observed result, and limitation. See `docs/START_HERE.md`.